In [2]:
import sqlite3
import pandas as pd
import numpy as np
import logging
import time
from ingestion_db import ingest_db

#logging config.

logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

#creating vendor summary.

def create_vendor_summary(conn):
    """
    Creates vendor-level summary with aligned aggregation grain
    """
    start = time.time()

    query = """
    WITH FreightSummary AS (
        SELECT
            VendorNumber,
            SUM(Freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber
    ),

    PurchaseSummary AS (
        SELECT
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Volume,
            pp.Price as Actual_price,
            SUM(p.Quantity) AS TotalPurchaseQuantity,
            SUM(p.Dollars) AS TotalPurchaseDollars
        from purchases as p
        join purchase_prices as pp
            on p.Brand = pp.Brand
        where p.PurchasePrice > 0
        Group by
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.price,
            pp.Volume
    ),

    SalesSummary AS (
        SELECT
            VendorNO AS VendorNumber,
            Brand,
            sum(SalesPrice) as TotalSalesPrice,
            SUM(SalesQuantity) AS TotalSalesQuantity,
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY VendorNO, Brand
    )
SELECT
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.Volume,
    ps.Actual_price,
    ss.TotalSalesPrice,
    ps.TotalPurchaseQuantity,
    ps.TotalPurchaseDollars,
    ss.TotalSalesQuantity,
    ss.TotalSalesDollars,
    ss.TotalExciseTax,
    fs.FreightCost
    
FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss

    ON ps.VendorNumber = ss.VendorNumber
   AND ss.Brand = ps.Brand
LEFT JOIN FreightSummary fs
    ON ss.VendorNumber = fs.VendorNumber
    """

    df = pd.read_sql_query(query, conn)

    logging.info(f"Vendor summary created: {df.shape}")
    logging.info(f"Time taken: {(time.time() - start)/60:.2f} minutes")

    return df


#data cleaning and kpi

def clean_data(df):
    start = time.time()

    df['Volume'] = df['Volume'].astype('float64')
    df.fillna(0, inplace=True)

    df['VendorName'] = df['VendorName'].astype(str).str.strip()
    df['Description'] = df['Description'].astype(str).str.strip()

    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    df['ProfitMargin'] = df['GrossProfit'] / df['TotalSalesDollars'] * 100
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
    df['SalesToPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']

    logging.info(f"Data cleaned: {df.shape}")
    logging.info(f"Cleaning time: {(time.time() - start)/60:.2f} minutes")

    return df

#main execution 

if __name__ == "__main__":

    conn = sqlite3.connect("inventory_db")

    logging.info("Creating vendor summary...")
    summary_df = create_vendor_summary(conn)

    logging.info("Cleaning data...")
    clean_df = clean_data(summary_df)

    logging.info(f"Final rows after filter: {clean_df.shape}")

    logging.info("Ingesting data...")
    ingest_db(clean_df, "vendor_sales_summary", conn)

    logging.info("Pipeline completed successfully")


In [3]:
clean_df.shape

(10692, 18)

In [4]:
(clean_df == 0).sum()


VendorNumber               0
VendorName                 0
Brand                      0
Description                0
PurchasePrice              0
Volume                     0
Actual_price               0
TotalSalesPrice          178
TotalPurchaseQuantity      0
TotalPurchaseDollars       0
TotalSalesQuantity       178
TotalSalesDollars        178
TotalExciseTax           178
FreightCost              178
GrossProfit                1
ProfitMargin               1
StockTurnover            178
SalesToPurchaseRatio     178
dtype: int64

In [6]:
clean_df.isnull().sum()

VendorNumber             0
VendorName               0
Brand                    0
Description              0
PurchasePrice            0
Volume                   0
Actual_price             0
TotalSalesPrice          0
TotalPurchaseQuantity    0
TotalPurchaseDollars     0
TotalSalesQuantity       0
TotalSalesDollars        0
TotalExciseTax           0
FreightCost              0
GrossProfit              0
ProfitMargin             0
StockTurnover            0
SalesToPurchaseRatio     0
dtype: int64

In [7]:
df= pd.read_sql_query("""select * from vendor_sales_summary where GrossProfit > 0
and ProfitMargin > 0 and TotalsalesQuantity > 0""",conn)

In [8]:
df.shape

(8564, 18)

In [9]:
df.to_csv("vendor_sales_summary.csv", index=False)